# Modeling за Flickr8k

Този notebook съдържа подготовката на данните и следващите стъпки за обучение на image-captioning модела.

## 0a. Разучаване на техники и научни подходи

Преди моделирането разглеждам задачата като encoder-decoder проблем. Изображението се кодира до числово представяне, а decoder-ът генерира caption дума по дума.

Основните техники, които използвам като ориентир, са:

- CNN + RNN/LSTM архитектура: CNN извлича visual features, а LSTM генерира текста;
- attention механизъм: decoder-ът може да се фокусира върху различни части от изображението;
- transformer-based модели: по-съвременен подход, при който attention е основният механизъм;
- retrieval/baseline подходи: избор на caption чрез близост до вече съществуващи captions.

За реализацията използвам VGG16 + LSTM, защото това е ясен baseline, който позволява да се проследи целият работен процес: preprocessing, vocabulary, training, evaluation и Streamlit интерфейс.


## 0. Настройка и зареждане на dataset-а

Избери kernel `Python (.venv) — Flickr8k Captioning`.

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "DATA" / "flickr8k").exists()
)
DATA_DIR = PROJECT_ROOT / "DATA" / "flickr8k"
IMAGES_DIR = DATA_DIR / "Images"
CAPTIONS_PATH = DATA_DIR / "captions.txt"
SPLITS_DIR = DATA_DIR / "splits"
TORCH_CACHE_DIR = PROJECT_ROOT / ".torch"
os.environ.setdefault("TORCH_HOME", str(TORCH_CACHE_DIR))

if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Images directory not found: {IMAGES_DIR}")
if not CAPTIONS_PATH.is_file():
    raise FileNotFoundError(f"Captions file not found: {CAPTIONS_PATH}")

captions = (
    pd.read_csv(CAPTIONS_PATH, sep="|")
    .rename(columns={"image_name": "image", "caption_text": "caption"})
    [["image", "caption"]]
    .dropna()
    .reset_index(drop=True)
)

print(f"Captions: {len(captions):,}")
print(f"Уникални изображения: {captions['image'].nunique():,}")
display(captions.head())

Captions: 40,455
Уникални изображения: 8,091


,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


## 1. Train / validation / test split

Разделяме по **уникални изображения**, за да останат всички captions на едно изображение в един split. Използваме детерминиран split `80% / 10% / 10%` със seed `42`.

In [2]:
SPLIT_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10

unique_images = np.array(sorted(captions["image"].unique()))
rng = np.random.default_rng(SPLIT_SEED)
shuffled_images = rng.permutation(unique_images)

train_end = round(len(shuffled_images) * TRAIN_RATIO)
val_end = train_end + round(len(shuffled_images) * VAL_RATIO)

train_images = set(shuffled_images[:train_end])
val_images = set(shuffled_images[train_end:val_end])
test_images = set(shuffled_images[val_end:])

train_captions = captions[captions["image"].isin(train_images)].copy().reset_index(drop=True)
val_captions = captions[captions["image"].isin(val_images)].copy().reset_index(drop=True)
test_captions = captions[captions["image"].isin(test_images)].copy().reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "images": [len(train_images), len(val_images), len(test_images)],
    "caption_rows": [len(train_captions), len(val_captions), len(test_captions)],
})
split_summary["image_share"] = split_summary["images"] / len(unique_images)
split_summary["caption_share"] = split_summary["caption_rows"] / len(captions)

display(split_summary)

,split,images,caption_rows,image_share,caption_share
0,train,6473,32365,0.800025,0.800025
1,validation,809,4045,0.099988,0.099988
2,test,809,4045,0.099988,0.099988


In [3]:
assert train_images.isdisjoint(val_images)
assert train_images.isdisjoint(test_images)
assert val_images.isdisjoint(test_images)
assert train_images | val_images | test_images == set(unique_images)
assert len(train_captions) + len(val_captions) + len(test_captions) == len(captions)

print("✓ Няма data leakage между split-овете.")
print("✓ Всички изображения и captions са разпределени.")

✓ Няма data leakage между split-овете.
✓ Всички изображения и captions са разпределени.


In [4]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

train_captions.to_csv(SPLITS_DIR / "train.csv", index=False)
val_captions.to_csv(SPLITS_DIR / "validation.csv", index=False)
test_captions.to_csv(SPLITS_DIR / "test.csv", index=False)

print(f"Split файловете са записани в: {SPLITS_DIR}")

Split файловете са записани в: /Users/nxics/Downloads/DL_25-26-main 2/DATA/flickr8k/splits


## 2. Vocabulary

Изграждаме vocabulary **само от train split-а**, за да не допускаме информация от validation/test. Думи с честота под `5` се представят чрез `<unk>`.

In [ ]:
import sys

import torch
from torch.utils.data import DataLoader

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from projects.image_captioning.data import Flickr8kCaptionDataset, make_caption_collate_fn
from projects.image_captioning.transforms import get_eval_transforms, get_train_transforms
from projects.image_captioning.vocab import Vocabulary, tokenize

In [ ]:
MIN_WORD_FREQUENCY = 5

vocabulary = Vocabulary(min_freq=MIN_WORD_FREQUENCY)
vocabulary.fit(train_captions["caption"].tolist())

train_caption_lengths = train_captions["caption"].map(lambda text: len(tokenize(text)))
# +2 за <start> и <end>. 95-и percentile ограничава много дългите редки captions.
MAX_CAPTION_LENGTH = int(np.ceil(train_caption_lengths.quantile(0.95))) + 2

print(f"Vocabulary size: {len(vocabulary):,}")
print(f"Minimum word frequency: {MIN_WORD_FREQUENCY}")
print(f"Maximum encoded caption length: {MAX_CAPTION_LENGTH}")
print(f"PAD index: {vocabulary.pad_idx}")
print(f"START index: {vocabulary.start_idx}")
print(f"END index: {vocabulary.end_idx}")
print(f"UNK index: {vocabulary.unk_idx}")

In [ ]:
sample_caption = train_captions.iloc[0]["caption"]
sample_encoded = vocabulary.encode(sample_caption)

print("Caption:", sample_caption)
print("Encoded:", sample_encoded)
print("Decoded:", vocabulary.decode(sample_encoded))

## 3. PyTorch Dataset и DataLoader

Train split-ът използва augmentation, а validation/test използват само детерминиран preprocessing. `num_workers=0` е най-надеждната настройка вътре в notebook.

In [ ]:
IMAGE_SIZE = 224

train_dataset = Flickr8kCaptionDataset(
    IMAGES_DIR,
    train_captions,
    vocabulary,
    transform=get_train_transforms(IMAGE_SIZE),
    max_length=MAX_CAPTION_LENGTH,
)
val_dataset = Flickr8kCaptionDataset(
    IMAGES_DIR,
    val_captions,
    vocabulary,
    transform=get_eval_transforms(IMAGE_SIZE),
    max_length=MAX_CAPTION_LENGTH,
)
test_dataset = Flickr8kCaptionDataset(
    IMAGES_DIR,
    test_captions,
    vocabulary,
    transform=get_eval_transforms(IMAGE_SIZE),
    max_length=MAX_CAPTION_LENGTH,
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

collate_fn = make_caption_collate_fn(vocabulary.pad_idx)
loader_generator = torch.Generator().manual_seed(SPLIT_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
    generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)

print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
print(f"Test batches: {len(test_loader):,}")

In [ ]:
batch_images, batch_captions = next(iter(train_loader))

print("Image batch shape:", tuple(batch_images.shape))
print("Caption batch shape:", tuple(batch_captions.shape))
print("Image dtype:", batch_images.dtype)
print("Caption dtype:", batch_captions.dtype)
print("First decoded caption:", vocabulary.decode(batch_captions[0].tolist()))

assert batch_images.shape == (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
assert batch_captions.shape == (BATCH_SIZE, MAX_CAPTION_LENGTH)
assert batch_captions.dtype == torch.long
print("✓ DataLoader batch-ът е готов за модела.")

## 4. VGG16 encoder + LSTM decoder

Използваме предварително обучен VGG16 като image encoder и LSTM като text decoder. Convolutional слоевете на VGG16 са замразени в началото, а projection слоят и decoder-ът се обучават.

In [ ]:
from torch import nn

from projects.image_captioning.model import CaptioningModel

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Training device:", DEVICE)

In [ ]:
EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
LEARNING_RATE = 1e-3

# При първото изпълнение torchvision може да свали VGG16 weights (~528 MB).
torch.manual_seed(SPLIT_SEED)

model = CaptioningModel(
    vocab_size=len(vocabulary),
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    freeze_encoder=True,
    pretrained_encoder=True,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=vocabulary.pad_idx)
trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.Adam(trainable_parameters, lr=LEARNING_RATE)

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameter_count = sum(parameter.numel() for parameter in trainable_parameters)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameter_count:,}")
print(f"Frozen parameters: {total_parameters - trainable_parameter_count:,}")

## 5. Кеширане на VGG features

VGG backbone-ът е замразен, затова извличаме pooled features само веднъж за всяко уникално изображение. Projection слоят и LSTM остават обучаеми. Това премахва най-тежката част от всяка training епоха.

`FEATURE_CACHE_MODE = "smoke"` кешира по 32 изображения от split. За реално обучение използвай `"full"`.

In [ ]:
from projects.image_captioning.data import Flickr8kFeatureCaptionDataset
from projects.image_captioning.features import cache_vgg_backbone_features

FEATURE_CACHE_MODE = "full"  # cache-ът вече е изграден за всички изображения
FEATURES_DIR = DATA_DIR / "features" / "vgg16_pool7"
cache_limit = 32 if FEATURE_CACHE_MODE == "smoke" else None

for split_name, split_frame in [
    ("train", train_captions),
    ("validation", val_captions),
    ("test", test_captions),
]:
    cache_summary = cache_vgg_backbone_features(
        model.encoder,
        IMAGES_DIR,
        split_frame,
        FEATURES_DIR,
        get_eval_transforms(IMAGE_SIZE),
        device=DEVICE,
        batch_size=16,
        num_workers=0,
        max_images=cache_limit,
    )
    print(split_name, cache_summary)

print("Feature cache:", FEATURES_DIR)

In [ ]:
def frame_with_cached_features(frame):
    available_images = {
        image_name
        for image_name in frame["image"].unique()
        if (FEATURES_DIR / f"{Path(image_name).stem}.pt").is_file()
    }
    return frame[frame["image"].isin(available_images)].copy().reset_index(drop=True)


if FEATURE_CACHE_MODE == "smoke":
    active_train_captions = frame_with_cached_features(train_captions)
    active_val_captions = frame_with_cached_features(val_captions)
    active_test_captions = frame_with_cached_features(test_captions)
else:
    active_train_captions = train_captions
    active_val_captions = val_captions
    active_test_captions = test_captions

train_feature_dataset = Flickr8kFeatureCaptionDataset(
    FEATURES_DIR, active_train_captions, vocabulary, MAX_CAPTION_LENGTH
)
val_feature_dataset = Flickr8kFeatureCaptionDataset(
    FEATURES_DIR, active_val_captions, vocabulary, MAX_CAPTION_LENGTH
)
test_feature_dataset = Flickr8kFeatureCaptionDataset(
    FEATURES_DIR, active_test_captions, vocabulary, MAX_CAPTION_LENGTH
)

train_loader = DataLoader(
    train_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
    generator=torch.Generator().manual_seed(SPLIT_SEED),
)
val_loader = DataLoader(
    val_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_feature_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

batch_images, batch_captions = next(iter(train_loader))
print("Cached feature batch:", tuple(batch_images.shape))
print("Train/val/test rows:", len(train_feature_dataset), len(val_feature_dataset), len(test_feature_dataset))

## 6. Forward pass и loss проверка

Decoder input е caption без последния token, а target е caption без `<start>`. Padding позициите се игнорират от loss функцията.

In [ ]:
# Малък batch е достатъчен за sanity check и пази памет при CPU изпълнение.
sanity_images = batch_images[:2].to(DEVICE)
sanity_captions = batch_captions[:2].to(DEVICE)

model.train()
optimizer.zero_grad(set_to_none=True)

decoder_inputs = sanity_captions[:, :-1]
targets = sanity_captions[:, 1:]
logits = model(sanity_images, decoder_inputs)

loss = criterion(logits.reshape(-1, len(vocabulary)), targets.reshape(-1))

print("Decoder input shape:", tuple(decoder_inputs.shape))
print("Logits shape:", tuple(logits.shape))
print("Target shape:", tuple(targets.shape))
print("Initial loss:", float(loss.detach().cpu()))

assert logits.shape[:2] == targets.shape
assert logits.shape[-1] == len(vocabulary)
assert torch.isfinite(loss)
print("✓ Forward pass и loss са коректни.")

## 7. Training и validation loop

Цикълът следи train/validation loss, прилага gradient clipping и запазва checkpoint при най-добър validation loss.

`TRAINING_MODE = "smoke"` изпълнява само по 1 batch за техническа проверка. `"full"`, когато искаме реалното обучение.

In [ ]:
from projects.image_captioning.training import run_caption_epoch, save_caption_checkpoint

TRAINING_MODE = "full"
NUM_EPOCHS = 5
GRADIENT_CLIP = 5.0

if TRAINING_MODE == "smoke":
    epochs_to_run = 1
    max_train_batches = 1
    max_validation_batches = 1
    checkpoint_path = PROJECT_ROOT / "checkpoints" / "smoke_test_model.pt"
elif TRAINING_MODE == "full":
    epochs_to_run = NUM_EPOCHS
    max_train_batches = None
    max_validation_batches = None
    checkpoint_path = PROJECT_ROOT / "checkpoints" / "best_model.pt"
else:
    raise ValueError('TRAINING_MODE трябва да бъде "smoke" или "full"')

print("Training mode:", TRAINING_MODE)
print("Checkpoint path:", checkpoint_path)

In [ ]:
MODEL_CONFIG = {
    "embed_size": EMBED_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "freeze_encoder": True,
}

history = []
best_validation_loss = float("inf")

for epoch in range(1, epochs_to_run + 1):
    train_loss = run_caption_epoch(
        model,
        train_loader,
        criterion,
        DEVICE,
        optimizer=optimizer,
        gradient_clip=GRADIENT_CLIP,
        max_batches=max_train_batches,
        log_every=100 if TRAINING_MODE == "full" else None,
    )
    validation_loss = run_caption_epoch(
        model,
        val_loader,
        criterion,
        DEVICE,
        max_batches=max_validation_batches,
    )

    epoch_metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
    }
    history.append(epoch_metrics)

    print(
        f"Epoch {epoch:02d}/{epochs_to_run:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"validation loss: {validation_loss:.4f}"
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        save_caption_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            vocabulary,
            epoch=epoch,
            validation_loss=validation_loss,
            model_config=MODEL_CONFIG,
            max_caption_length=MAX_CAPTION_LENGTH,
            history=history,
        )
        print(f"  ✓ Saved checkpoint: {checkpoint_path}")

In [ ]:
import matplotlib.pyplot as plt

history_frame = pd.DataFrame(history)
display(history_frame)

if not history_frame.empty:
    axis = history_frame.plot(
        x="epoch",
        y=["train_loss", "validation_loss"],
        marker="o",
        title="Training history",
        grid=True,
    )
    axis.set_ylabel("Cross-entropy loss")
    plt.tight_layout()
    plt.show()

## 7a. Resume training с early stopping

След ръчна проверка на резултатите продължих VGG16 + LSTM модела от `checkpoints/best_model.pt` чрез `scripts/resume_caption_training.py`. Целта беше да проверя дали повече обучение подобрява VGG16 + LSTM модела, без да презаписвам с overfit-нат checkpoint.

Използвах:

- resume от epoch 4;
- learning rate `3e-4`;
- `ReduceLROnPlateau`;
- early stopping;
- кеширани VGG features.

Най-добрият checkpoint след resume training е epoch 5 с validation loss `2.7857`. След това train loss продължи да пада, но validation loss се влошаваше, което показва overfitting. Затова early stopping спря training-а на epoch 10 и запазих най-добрия checkpoint от epoch 5.

Обновени test резултати:

| Метрика | Стойност |
|---|---:|
| test loss | 2.7643 |
| BLEU-1 | 0.5386 |
| BLEU-4 | 0.1392 |
| ROUGE-L | 0.3918 |


## 8. Evaluation и caption generation

Зареждаме checkpoint-а, измерваме test loss и генерираме по един caption за уникално test изображение. Сравняваме го с всичките 5 референтни captions чрез BLEU-1, BLEU-4 и ROUGE-L.

In [ ]:
from PIL import Image

from projects.image_captioning.evaluation import (
    corpus_bleu,
    load_caption_checkpoint,
    rouge_l_f1,
)
from projects.image_captioning.inference import generate_caption_from_backbone_features

EVALUATION_MODE = "full"
best_checkpoint_path = PROJECT_ROOT / "checkpoints" / "best_model.pt"
smoke_checkpoint_path = PROJECT_ROOT / "checkpoints" / "smoke_test_model.pt"
evaluation_checkpoint_path = (
    best_checkpoint_path if best_checkpoint_path.exists() else smoke_checkpoint_path
)

if not evaluation_checkpoint_path.exists():
    raise FileNotFoundError("Първо изпълни training секцията, за да създадеш checkpoint.")

evaluation_model, evaluation_vocabulary, evaluation_checkpoint = load_caption_checkpoint(
    evaluation_checkpoint_path,
    DEVICE,
)
evaluation_max_length = evaluation_checkpoint.get("max_caption_length", 30)

print("Checkpoint:", evaluation_checkpoint_path)
print("Epoch:", evaluation_checkpoint.get("epoch"))
print("Validation loss:", evaluation_checkpoint.get("validation_loss"))

In [ ]:
test_loss = run_caption_epoch(
    evaluation_model,
    test_loader,
    criterion,
    DEVICE,
    max_batches=1 if EVALUATION_MODE == "smoke" else None,
)
print(f"Test loss ({EVALUATION_MODE}): {test_loss:.4f}")

In [ ]:
grouped_test_captions = test_captions.groupby("image")["caption"].apply(list)
evaluation_image_names = sorted(grouped_test_captions.index)
if EVALUATION_MODE == "smoke":
    evaluation_image_names = evaluation_image_names[:5]

evaluation_rows = []
metric_references = []
metric_hypotheses = []

for image_name in evaluation_image_names:
    references = grouped_test_captions[image_name]
    cached_features = torch.load(
        FEATURES_DIR / f"{Path(image_name).stem}.pt",
        map_location="cpu",
        weights_only=True,
    )
    generated_caption = generate_caption_from_backbone_features(
        evaluation_model,
        cached_features,
        evaluation_vocabulary,
        max_length=evaluation_max_length,
        device=DEVICE,
    )

    metric_references.append([tokenize(reference) for reference in references])
    metric_hypotheses.append(tokenize(generated_caption))
    evaluation_rows.append({
        "image": image_name,
        "generated_caption": generated_caption,
        "reference_caption": references[0],
    })

evaluation_results = pd.DataFrame(evaluation_rows)
print(f"Generated captions: {len(evaluation_results):,}")
display(evaluation_results.head(20))

In [ ]:
caption_metrics = pd.Series({
    "evaluated_images": len(evaluation_image_names),
    "BLEU-1": corpus_bleu(metric_references, metric_hypotheses, max_order=1),
    "BLEU-4": corpus_bleu(metric_references, metric_hypotheses, max_order=4),
    "ROUGE-L": rouge_l_f1(metric_references, metric_hypotheses),
    "test_loss": test_loss,
}, name="value")

display(caption_metrics.to_frame())

REPORTS_DIR = PROJECT_ROOT / "reports" / "model"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
evaluation_results.to_csv(REPORTS_DIR / "generated_captions.csv", index=False)
caption_metrics.to_frame().to_csv(REPORTS_DIR / "evaluation_metrics.csv")
print("Evaluation reports:", REPORTS_DIR)

## 9. Streamlit приложение

Приложението има три режима:

- `Project VGG retrieval` — използва VGG features и връща caption от визуално близко training изображение;
- `Project baseline VGG16 + LSTM` — използва обучения в проекта модел от `checkpoints/best_model.pt`;
- `Pretrained BLIP` — използва предварително обучен vision-language модел за сравнение.

Стартиране от root директорията:

```bash
source .venv/bin/activate
python -m streamlit run streamlit_app/app.py
```

Ако `python` не е наличен като команда, може директно:

```bash
.venv/bin/python -m streamlit run streamlit_app/app.py
```

VGG16 + LSTM режимът показва основния генеративен модел, а VGG retrieval режимът служи като по-заземен project вариант при изображения, за които LSTM decoder-ът дава твърде общи captions. BLIP режимът е сравнение с по-силен предварително обучен модел.

## 9a. Greedy decoding и beam search

При inference използвам два режима за генериране на caption:

- `greedy` — на всяка стъпка избира най-вероятната следваща дума;
- `beam` — държи няколко възможни partial captions и избира най-добрия цялостен кандидат.

Beam search може да направи captions по-стабилни, но не променя самия модел. Ако входната снимка е много различна от разпределението на Flickr8k, резултатът пак може да бъде generic или неточен.


## 10. Ограничения на baseline модела

Този модел не е универсален описвач на всякакви изображения. Той е baseline, обучен върху Flickr8k, затова работи най-добре върху снимки, подобни на dataset-а: хора навън, кучета, деца, спорт и прости outdoor сцени.

При произволни лични снимки — например screenshot, храна, стая, документ, предмет на бюро или много специфична сцена — моделът може да генерира неточно описание. Това е очаквано ограничение на dataset-а и архитектурата, не задължително грешка в кода.

За значително по-добро качество върху произволни снимки следващите варианти са attention decoder, по-голям dataset като MS COCO или pretrained vision-language модел.


## 10a. Pretrained BLIP режим

Тъй като VGG16 + LSTM моделът е обучен само върху Flickr8k, той може да генерира неточни captions при произволни снимки. Затова в Streamlit включвам отделен `Pretrained BLIP` режим за сравнение с предварително обучен vision-language модел.

Важно е режимите да се разграничават:

- VGG16 + LSTM е моделът, който обучавам в проекта;
- VGG retrieval е проектен вариант, който използва VGG features и train captions;
- BLIP е предварително обучен модел, използван само за сравнение с по-съвременен подход.

Така в проекта се вижда разликата между собствено обучен модел, retrieval подход и предварително обучен vision-language модел.


## 10b. VGG retrieval baseline

След ръчна проверка на резултатите се вижда, че VGG16 + LSTM моделът понякога генерира често срещани, но визуално неточни captions. Затова използвам и `Project VGG retrieval` режим.

Идеята е проста:

1. извличам VGG features за входното изображение;
2. сравнявам ги с кешираните VGG features на train изображенията;
3. намирам най-близките training примери;
4. връщам caption от най-близкото изображение.

Това не е толкова гъвкаво като LSTM decoder, но е по-grounded: caption-ът идва от визуално подобен пример, вместо да бъде изцяло генериран от езиковия prior на decoder-а. За малък dataset като Flickr8k това е силен и честен baseline.


## 10c. TF-IDF caption experiment

За изискването за word embedding/text representation направих TF-IDF caption experiment. Той не е основният sequence decoder, но показва кои думи са най-характерни за train captions и помага да се обясни dataset bias-ът.

Най-силните TF-IDF думи след премахване на stop words са: `dog`, `man`, `two`, `black`, `white`, `boy`, `woman`, `girl`, `wearing`, `people`, `water`.

Това потвърждава, че Flickr8k съдържа много captions за хора, кучета, цветове, движение и outdoor сцени.


## 10d. Architecture experiments

Изпълних кратки architecture experiments върху кеширани VGG features чрез `scripts/run_architecture_experiments.py`.

| Вариант | Hidden size | Layers | Dropout | Validation loss | Sample BLEU-4 |
|---|---:|---:|---:|---:|---:|
| `small_lstm_256` | 256 | 1 | 0.0 | 3.1199 | 0.0921 |
| `baseline_lstm_512` | 512 | 1 | 0.0 | 2.9511 | 0.1042 |
| `two_layer_lstm_dropout` | 512 | 2 | 0.3 | 3.0947 | 0.0892 |

При еднакъв кратък training budget най-добър е `baseline_lstm_512`. Това подкрепя избора на основния модел.


## 10e. Qualitative comparison report

Използвам `scripts/build_qualitative_report.py`, за да сравня reference captions, VGG16+LSTM caption и VGG retrieval caption за избрани test изображения.

Този отчет е важен, защото автоматичните метрики не винаги показват дали caption-ът е визуално смислен. Ръчната проверка показва кога LSTM decoder-ът дава неточно описание и кога retrieval вариантът е по-заземен.


## 11. Обобщение на експериментите спрямо изискванията

| Група | Реализирано в проекта | Следващ експеримент |
|---|---|---|
| Preprocessing | `Resize(224, 224)`, ImageNet normalization, train augmentation | center crop, color jitter или различен image size |
| Augmentation | horizontal flip + rotation само за train | по-силни image transformations и сравнение по validation loss |
| Word/text representations | обучаем `nn.Embedding` слой + TF-IDF caption experiment | GloVe, Word2Vec, FastText или transformer text encoder |
| Architecture | frozen VGG16 + LSTM, hidden size/layers/dropout experiments | attention decoder или частично размразен VGG16 |
| Evaluation | validation loss, test loss, BLEU-1, BLEU-4, ROUGE-L, qualitative comparison | повече ръчни примери и сравнение с MS COCO модел |
| UI | Streamlit приложение с VGG retrieval, VGG16 + LSTM и BLIP режим | по-подробна визуализация на nearest neighbors |
| Testing | тестове за data, vocab, model, training, evaluation, retrieval, scripts и presentation builder | end-to-end Streamlit smoke test |

Основният резултат е VGG16 + LSTM baseline. Той служи като проследима собствена реализация, а VGG retrieval и BLIP режимите помагат да се обясни защо по-силни или по-заземени подходи дават по-надеждни captions при произволни изображения.

## 11a. Таблица с експериментални конфигурации

За да държа вариантите подредени, използвам модул с experiment configs. Така основният модел и следващите варианти са описани едновременно в кода и в доклада.


In [ ]:
from projects.image_captioning.experiments import summarize_experiment_configs

experiment_configs_df = pd.DataFrame(summarize_experiment_configs())
experiment_configs_df


## 11b. Training reliability проверки

След qualitative проверката с реални изображения добавям няколко reliability подобрения. Те не правят baseline модела автоматично по-силен, но помагат да се провери дали проблемът е в setup-а или в ограниченията на dataset-а и архитектурата.

- `TrainingConfig` описва epochs, learning rate, gradient clipping, scheduler и early stopping;
- `EarlyStopping` спира обучението, ако validation loss не се подобрява;
- `ReduceLROnPlateau` намалява learning rate при застой;
- `describe_image_preprocessing` описва resize и ImageNet normalization, за да е ясно, че train/eval preprocessing са съвместими;
- `scripts/diagnose_captioning_setup.py` проверява dataset, checkpoint и preprocessing metadata.


In [ ]:
from projects.image_captioning.training import TrainingConfig
from projects.image_captioning.transforms import describe_image_preprocessing

training_config = TrainingConfig(epochs=20, learning_rate=1e-3, early_stopping_patience=5)
preprocessing_config = describe_image_preprocessing(IMAGE_SIZE)

training_config, preprocessing_config


## 12. Финално състояние на проекта

Към текущата версия проектът покрива основния цикъл за image captioning:

- EDA notebook с анализ на captions, изображения, аномалии и визуализации;
- train/validation/test split по уникални изображения;
- vocabulary, dataset класове и DataLoader-и;
- VGG16 encoder + LSTM decoder baseline;
- кеширани VGG features за по-бързо обучение;
- resume training с scheduler и early stopping;
- evaluation с test loss, BLEU-1, BLEU-4 и ROUGE-L;
- TF-IDF text experiment;
- architecture experiments за LSTM варианти;
- qualitative comparison report;
- Streamlit приложение с три режима;
- автоматични тестове;
- PowerPoint презентация в `presentation/image_captioning_presentation.pptx`.

Последни записани резултати на основния checkpoint:

| Метрика | Стойност |
|---|---:|
| validation loss | 2.7857 |
| test loss | 2.7643 |
| BLEU-1 | 0.5386 |
| BLEU-4 | 0.1392 |
| ROUGE-L | 0.3918 |

Последната пълна проверка на тестовете е `31 passed`. Ако проектът се представя на живо, най-сигурният режим за произволни изображения е `Project VGG retrieval`, а `Project baseline VGG16 + LSTM` трябва да се показва като собствен baseline с ясно описани ограничения.